# **Notebook 4: RAG Implementation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] `corporate_policies/` folder with `.md` SOP files
- [ ] `outputs.json` — Created by Notebook 3
- [ ] GPU runtime enabled

**Files this notebook will CREATE:**
- [ ] `./chroma_db/` — Persisted ChromaDB vector index _(Required by NB5 and NB7)_
- [ ] `outputs.json` (updated) — adds `naive_rag_output` _(Required by NB5 and NB7)_

---

### **Task 3.2: Implement Retrieval-Assisted Generation**

#### **3.2.1 Generate Embeddings [4 marks]**
**The Task:** Initialise the `all-MiniLM-L6-v2` embedding model, embed the SOP documents, and validate the embeddings were produced.

**Hints & Tips:**
* Load the SOP documents with `TextLoader` first (or reuse the corpus from NB2).
* `HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")` runs on CPU — no GPU needed.
* Validate by embedding one sample string and checking the vector length (384 dims for MiniLM).
* You MUST use the same embedding model when reloading in Notebooks 5 and 7.

**Embedding Model Options:**
* **`all-MiniLM-L6-v2`** (recommended): 384-dim, fast, ~80MB.
* **`all-mpnet-base-v2`**: 768-dim, higher quality, slower.
* **`bge-small-en-v1.5`**: 384-dim, newer architecture.

**Learner Inference:** Your text is now coordinates in semantic space — similar meanings sit close together.

In [1]:
# YOUR CODE HERE

import os
from glob import glob
from langchain_community.document_loaders import TextLoader

SOP_DIR = "data/Dataset/sop_documents"

sop_files = glob(os.path.join(SOP_DIR, "*.md"))

documents = []

for file_path in sop_files:
    loader = TextLoader(file_path, encoding="utf-8")
    documents.extend(loader.load())

print("Total SOP documents loaded:", len(documents))
print("First document:", documents[0].metadata["source"])


C:\Users\divya\AppData\Local\Temp\ipykernel_32892\2179406152.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


Total SOP documents loaded: 13
First document: data/Dataset/sop_documents\account_recovery.md


In [2]:

from langchain_huggingface import HuggingFaceEmbeddings

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"}
)

print("Embedding model loaded successfully!")


W1121 14:19:54.443000 32892 .venv\Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
W1121 14:19:55.557000 32892 .venv\Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully!


In [3]:
# check that the embedding model really produces numbers
sample_vector = embeddings.embed_query("Where is my refund?")
print("Vector length:", len(sample_vector))
print("First 5 numbers:", sample_vector[:5])

# embed all the SOP documents
doc_vectors = embeddings.embed_documents([d.page_content for d in documents])
print("Documents embedded:", len(doc_vectors))
print("Each document has", len(doc_vectors[0]), "numbers")

Vector length: 384
First 5 numbers: [-0.03488024324178696, -0.006559434812515974, 0.0374530591070652, 0.04802258312702179, 0.046198319643735886]
Documents embedded: 13
Each document has 384 numbers


Embedding validated: each text becomes a 384-number vector, and all 13 SOP documents were embedded.

#### **3.2.2 Build Vector Index [4 marks]**
**The Task:** Create a persistent Chroma vector index from the embedded documents, configure similarity search, and validate the index.

**Hints & Tips:**
* `Chroma.from_documents(docs, embeddings, persist_directory="./chroma_db")` auto-saves — no manual `.persist()` needed.
* Validate with `vector_db._collection.count()` — should equal the number of SOP documents.
* Run one test `.similarity_search("refund", k=1)` to confirm retrieval works.

**Vector DB Options:**
* **ChromaDB** (recommended): simple API, auto-persistence, LangChain integration.
* **FAISS**: faster for >100K docs, but no built-in persistence (manual serialization).

**Learner Inference:** The index lets you search by meaning — it returns the document mathematically closest to your query's coordinates.

In [14]:
from langchain_chroma import Chroma

# Start from an empty database every time, so re-running never adds duplicates
try:
    Chroma(persist_directory="./chroma_db", embedding_function=embeddings).delete_collection()
except Exception:
    pass   # nothing to delete on the very first run

vector_db = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

print("Chroma vector database created successfully!")

Chroma vector database created successfully!


In [15]:

print("Total vectors stored:", vector_db._collection.count())


Total vectors stored: 13


In [6]:

results = vector_db.similarity_search("refund", k=1)

print("Retrieved document:", results[0].metadata["source"])
print("\nDocument content:")
print(results[0].page_content[:500])


Retrieved document: data/Dataset/sop_documents\refund_policy.md

Document content:
# Refund Policy

## Eligibility
Customers may request a refund within 30 days of the original purchase or
delivery date, whichever is later. To be eligible, the item must be unused or
defective, and the request must reference a valid order identifier. Digital
goods and subscription fees already consumed are non-refundable except where
required by local consumer law.

## Refund Methods
Approved refunds are issued to the original payment method. If the original
card or wallet is no longer active, 


#### **3.2.3 Implement Retrieval Workflow [4 marks]**
**The Task:** Execute a "Naive RAG" workflow — pass the raw customer query into the vector DB, fetch the top result, and augment the LLM prompt.

**Hints & Tips:**
* Use `.similarity_search(query, k=1)` for the top-1 document.
* Check whether the raw query retrieved the WRONG policy — common with ambiguous queries.
* Inject context via the system prompt: `"Answer strictly using this SOP: {context}"`.

**Parameter Tuning:**
* `k=1`: one document (focused). `k=3`: more context if SOPs overlap. `k=5`: max, risks long prompts.

**Learner Inference:** Noisy queries often retrieve the wrong document — proving Naive RAG is flawed and motivating the fine-tuned router.

In [7]:
# YOUR CODE HERE
# Load the saved baseline and Naive RAG results from outputs.json.
# We will use the customer query and model responses for evaluation.

import json
with open("outputs.json", "r", encoding="utf-8") as f:
    baseline_data = json.load(f)

test_query = baseline_data["test_query"]

print("Customer Query:", test_query)


Customer Query: i need information about delivery periods


In [8]:

# Retrieve the most relevant SOP document from ChromaDB
# using the customer's query.
# We use k=1 to retrieve only the top matching document.

retrieved_docs = vector_db.similarity_search(test_query, k=1)

context = retrieved_docs[0].page_content

print("Customer Query:", test_query)
print("\nRetrieved SOP:", retrieved_docs[0].metadata["source"])
print("\nRetrieved Context:\n")
print(context)


Customer Query: i need information about delivery periods

Retrieved SOP: data/Dataset/sop_documents\order_tracking.md

Retrieved Context:

# Order Tracking

## Scope
Helps customers locate an order and interpret its status. Orders that have
clearly exceeded their delivery window follow the shipping delays procedure.

## Order States
- **Processing:** payment confirmed, not yet packed.
- **Preparing to ship:** packed, awaiting carrier pickup.
- **In transit:** carrier has the parcel; tracking events update as it moves.
- **Out for delivery:** on the vehicle for delivery that day.
- **Delivered:** carrier marked the parcel delivered.
- **Exception:** address issue, failed delivery, or held parcel needing
  action.

## Finding The Order
Locate the order by its identifier and the registered email. Provide the
tracking number and carrier link. Tracking can take up to 24 hours to show
the first scan after a label is created, so a label with no movement on day
one is expected, not lost.

## 

In [9]:
# Create a system prompt that provides the retrieved SOP as context.
# This instructs Qwen to answer using the company's policy
# and avoid inventing information.
system_prompt = f"""You are a customer support assistant.
Answer strictly using this SOP:

{context}

If the SOP does not contain the answer, say that the information is not available.
Do not invent information."""

# Create the conversation messages for the Qwen model.
# The system message contains the SOP context.
# The user message contains the original customer query.

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": test_query}
]

# Display the augmented prompt and customer query
print("Augmented Prompt:\n")
print(system_prompt)
print("\nCustomer Query:", test_query)


Augmented Prompt:

You are a customer support assistant.
Answer strictly using this SOP:

# Order Tracking

## Scope
Helps customers locate an order and interpret its status. Orders that have
clearly exceeded their delivery window follow the shipping delays procedure.

## Order States
- **Processing:** payment confirmed, not yet packed.
- **Preparing to ship:** packed, awaiting carrier pickup.
- **In transit:** carrier has the parcel; tracking events update as it moves.
- **Out for delivery:** on the vehicle for delivery that day.
- **Delivered:** carrier marked the parcel delivered.
- **Exception:** address issue, failed delivery, or held parcel needing
  action.

## Finding The Order
Locate the order by its identifier and the registered email. Provide the
tracking number and carrier link. Tracking can take up to 24 hours to show
the first scan after a label is created, so a label with no movement on day
one is expected, not lost.

## "Delivered But Not Received"
1. Confirm the shippi

In [10]:
# Load the Qwen LLM and tokenizer.
# Same 4-bit nf4 settings as Notebook 3 (Baseline), so Baseline and Naive RAG
# are compared fairly and the only difference between them is the retrieved SOP.

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# 4-bit settings (same as Notebook 3)
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# load the model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map="auto"
)

print("Qwen model and tokenizer loaded successfully!")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen model and tokenizer loaded successfully!


In [11]:
# Generate a Naive RAG response using the retrieved SOP context.
# max_new_tokens=120 matches Notebook 3 (Baseline), so both systems
# are allowed the same answer length.

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# convert the prompt into tensors and move them to the model device
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# generate a deterministic response
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False
    )

# decode only the newly generated tokens
naive_rag_output = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("Naive RAG Response:\n")
print(naive_rag_output)

Naive RAG Response:

The delivery period varies depending on your location and the service provider. Please refer to the specific details provided by the shipping company you're working with. They will be able to give you accurate information regarding when your package should arrive at your destination.


---
## Save Artifacts for Downstream Notebooks

In [12]:
# YOUR CODE HERE

# Save the Naive RAG response for use in the downstream evaluation notebooks.

baseline_data["naive_rag_output"] = naive_rag_output

with open("outputs.json", "w", encoding="utf-8") as f:
    json.dump(baseline_data, f, indent=4, ensure_ascii=False)

print("Naive RAG output saved successfully!")


Naive RAG output saved successfully!


In [13]:
# Verify that the persistent ChromaDB folder exists on disk.

import os

print("ChromaDB folder exists:", os.path.exists("./chroma_db"))

ChromaDB folder exists: True


### 3.2.3 Implement Retrieval Workflow — Summary and Conclusion

- **Customer Query:** The query was loaded from `outputs.json`: "i need information about delivery periods".

- **SOP Retrieval:** ChromaDB was searched with `similarity_search()` and `k=1`. The retrieved document was `order_tracking.md`. This is the **wrong SOP**: the question is about delivery timelines, which are covered in `shipping_delays.md`. The raw query shares words such as "delivery" with the order tracking SOP, so it was the closest match.

- **Retrieved Context:** The order tracking SOP explains order states, how to find an order, and what to do when a parcel is marked delivered but not received. It does not contain the standard delivery timelines.

- **Augmented Prompt:** The retrieved SOP was added to the system prompt, with instructions for Qwen to answer strictly from it and not to invent information. The original query was the user message.

- **Naive RAG Response:** The model said that the delivery period varies by location and service provider, and told the customer to check with the shipping company. It gave no delivery times.

- **Baseline Comparison:** The baseline model quoted delivery times from its own memory (for example "3-5 days" in the United States), which do not match the SOP. Naive RAG did not invent any numbers, so it is safer. But it also did not give the correct rule ("domestic orders deliver within 3–7 business days"), because the wrong SOP was retrieved.

**Conclusion:**

The Naive RAG workflow connected retrieval with answer generation, and it stopped the model inventing delivery times. However, searching with the raw customer message retrieved the wrong SOP, so the answer could not contain the information the customer needed. This shows that the quality of the answer depends on retrieving the correct document first. It motivates the fine-tuned intent router in the next stage, which identifies the intent (`delivery_period`) before searching.

---
## END-OF-NOTEBOOK CHECKLIST

> **IMPORTANT: Verify before proceeding to Notebook 5.**

- [ ] SOP documents loaded via `TextLoader`
- [ ] **Embeddings generated and validated** ← _Task 3.2.1_
- [ ] **ChromaDB index built, validated, and persisted** ← _Task 3.2.2_
- [ ] Naive similarity search executed on `test_query`
- [ ] Naive RAG output generated with SOP context injected
- [ ] **`./chroma_db/` exists on disk** ← _CRITICAL for NB5 and NB7_
- [ ] **`outputs.json` updated** with `naive_rag_output` ← _CRITICAL for NB5 and NB7_

**If any item is unchecked, fix it before moving on.**

## Notebook 4 — End-of-Notebook Checklist

Before proceeding to Notebook 5, verify the following:

- [x] SOP documents loaded successfully using `TextLoader` (13 documents).
- [x] Embeddings generated using `sentence-transformers/all-MiniLM-L6-v2`.
- [x] Embedding model loaded and validated (384 numbers per vector).
- [x] ChromaDB vector index created successfully.
- [x] Vector index validated with 13 stored documents.
- [x] Similarity search executed using `test_query`.
- [x] Naive search retrieved the wrong SOP (`order_tracking.md` instead of `shipping_delays.md`), showing the limit of Naive RAG.
- [x] Naive RAG response generated with the retrieved SOP injected into the prompt.
- [x] `./chroma_db/` folder existence verified on disk.
- [x] `outputs.json` updated successfully with the Naive RAG output.

**Important:** The ChromaDB folder and updated `outputs.json` are required by downstream notebooks, especially Notebooks 5 and 7.

All the above tasks have been completed and verified. Notebook 4 is ready for Notebook 5.